In [ ]:
import os
import time
import gc
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers.legacy import Adam as LegacyAdam
from tensorflow.keras.layers import (
    Input, LSTM, Bidirectional, Conv1D, Dense, 
    BatchNormalization, Activation, Dropout, 
    Flatten, MaxPooling1D, Concatenate,
    GlobalAveragePooling1D, GlobalMaxPooling1D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import backend as K

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [2]:
Yt = pd.read_csv('ws.csv', header=1, parse_dates=['Timestamp'])
Yt = Yt.rename(columns={'Timestamp': 'time'})

In [3]:
# wind direction to sin/cos
Yt['wind_sin'] = np.sin(np.deg2rad(Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360))
Yt['wind_cos'] = np.cos(np.deg2rad(Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360))

# SD turbulence columns
sd_cols = [
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s']

# Turbulence intensity TI = SD / mean
Yt['TI_110'] = Yt['Ch1_Anem_110.00m_E_SD_m/s'] / Yt['Ch1_Anem_110.00m_E_Avg_m/s']
Yt['TI_50']  = Yt['Ch2_Anem_50.00m_E_SD_m/s']  / Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['TI_30']  = Yt['Ch3_Anem_30.00m_E_SD_m/s']  / Yt['Ch3_Anem_30.00m_E_Avg_m/s']
Yt['TI_10']  = Yt['Ch4_Anem_10.00m_E_SD_m/s']  / Yt['Ch4_Anem_10.00m_E_Avg_m/s']

Yt.replace([np.inf, -np.inf], np.nan, inplace=True)
Yt.fillna(0, inplace=True)

# Gust deviation
Yt['gust_dev_110'] = Yt['Ch1_Anem_110.00m_E_Gust_m/s'] - Yt['Ch1_Anem_110.00m_E_Avg_m/s']
Yt['gust_dev_50']  = Yt['Ch2_Anem_50.00m_E_Gust_m/s']  - Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['gust_dev_30']  = Yt['Ch3_Anem_30.00m_E_Gust_m/s']  - Yt['Ch3_Anem_30.00m_E_Avg_m/s']
Yt['gust_dev_10']  = Yt['Ch4_Anem_10.00m_E_Gust_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']

# Vertical shear
Yt['shear_110_10'] = Yt['Ch1_Anem_110.00m_E_Avg_m/s'] - Yt['Ch4_Anem_10.00m_E_Avg_m/s']
Yt['shear_110_50'] = Yt['Ch1_Anem_110.00m_E_Avg_m/s'] - Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['shear_50_10']  = Yt['Ch2_Anem_50.00m_E_Avg_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']
Yt['shear_30_10']  = Yt['Ch3_Anem_30.00m_E_Avg_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']

# Temperature and pressure
Yt['temp'] = Yt['Ch9_Analog_10.00m_N_Avg_C']
Yt['pressure'] = Yt['Ch10_Analog_10.00m_N_Avg_kpa']

# Time cyclic features
Yt['minute'] = Yt['time'].dt.minute
Yt['minute_sin'] = np.sin(2 * np.pi * Yt['minute'] / 60)
Yt['minute_cos'] = np.cos(2 * np.pi * Yt['minute'] / 60)

# Base feature list
features = [
    # raw wind speeds
    'Ch4_Anem_10.00m_E_Avg_m/s',
    'Ch3_Anem_30.00m_E_Avg_m/s',
    'Ch2_Anem_50.00m_E_Avg_m/s',
    'Ch1_Anem_110.00m_E_Avg_m/s',
    # wind direction
    'wind_sin', 'wind_cos',
    # SD turbulence
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s',
    # TI
    'TI_110', 'TI_50', 'TI_30', 'TI_10',
    # gust deviation
    'gust_dev_110', 'gust_dev_50', 'gust_dev_30', 'gust_dev_10',
    # shear
    'shear_110_10', 'shear_110_50', 'shear_50_10', 'shear_30_10',
    # meteo
    'temp', 'pressure',
    # time
    'minute_sin', 'minute_cos']
target_h = 'Ch4_Anem_10.00m_E_Avg_m/s'

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

def make_xy(X_scaled, y_raw, window_size=12, pred_steps=1):
    X, y = [], []
    n = len(X_scaled)
    for i in range(n - window_size - pred_steps + 1):
        X.append(X_scaled[i : i + window_size, :])
        y.append(y_raw[i + window_size : i + window_size + pred_steps])
    return np.array(X), np.array(y)

def calculate_mape(true, pred):
    true = np.asarray(true)
    pred = np.asarray(pred)
    return np.mean(np.abs((true - pred) / np.maximum(np.abs(true), 1e-6))) * 100

f_list = [1, 2, 3, 4, 5, 6]
w_list = [3,6, 12,24]
target = target_h[0]

results = []
predictions = {}

df_y = Yt[['time'] + features].copy()
df_y = df_y.sort_values('time').reset_index(drop=True)

split_idx = int(0.8 * len(df_y))
train_df = df_y.iloc[:split_idx]
test_df  = df_y.iloc[split_idx:]

scaler_X = StandardScaler()
train_scaled = scaler_X.fit_transform(train_df[features].values)
test_scaled  = scaler_X.transform(test_df[features].values)

y_train_raw = train_df[target].values.astype(float)
y_test_raw  = test_df[target].values.astype(float)

for f in f_list:
    forecast_min = f * 10
    print(f"Forecast horizon: {forecast_min} min ({f} steps)")

    for w in w_list:
        print(f"\n>>> Training: Window Size={w}, Horizon={f} steps")
        start_time = time.time()

        X_train, y_train = make_xy(train_scaled, y_train_raw, w, f)
        X_test,  y_test  = make_xy(test_scaled,  y_test_raw,  w, f)

        scaler_y = MinMaxScaler()
        y_train_s = scaler_y.fit_transform(y_train.reshape(-1, 1)).reshape(-1, f)
        y_test_s  = scaler_y.transform(y_test.reshape(-1, 1)).reshape(-1, f)

        inp = Input(shape=(w, X_train.shape[2]))
        
        x = Conv1D(filters=64, kernel_size=3, padding="same", activation="relu")(inp)
        x = BatchNormalization()(x)
        x = MaxPooling1D(pool_size=2)(x)
        
        x = Bidirectional(LSTM(64, return_sequences=True))(x)
        x = Dropout(0.3)(x)

        x = Flatten()(x)
        x = Dense(64, activation="relu")(x)
        x = BatchNormalization()(x)
        x = Dropout(0.2)(x)

        out = Dense(f)(x)
        model = Model(inp, out)
        model.compile(optimizer=LegacyAdam(learning_rate=0.001), loss="huber")
        model.fit(
            X_train, y_train_s,
            epochs=150,
            batch_size=128,
            shuffle=False,
            validation_split=0.1,
            callbacks=[
                ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-5),
                EarlyStopping(patience=15, restore_best_weights=True)
            ],
            verbose=0)

        y_train_pred_s = model.predict(X_train, verbose=0)
        y_test_pred_s  = model.predict(X_test,  verbose=0)

        y_train_pred_real = scaler_y.inverse_transform(
            y_train_pred_s.reshape(-1, 1)
        ).reshape(-1, f)

        y_test_pred_real = scaler_y.inverse_transform(
            y_test_pred_s.reshape(-1, 1)
        ).reshape(-1, f)

        y_train_true_real = y_train
        y_test_true_real  = y_test

        flat_true_train = y_train_true_real.flatten()
        flat_pred_train = y_train_pred_real.flatten()
        flat_true_test  = y_test_true_real.flatten()
        flat_pred_test  = y_test_pred_real.flatten()

        rmse_train = np.sqrt(mean_squared_error(flat_true_train, flat_pred_train))
        rmse_test  = np.sqrt(mean_squared_error(flat_true_test, flat_pred_test))

        r2_train = r2_score(flat_true_train, flat_pred_train)
        r2_test  = r2_score(flat_true_test, flat_pred_test)

        mae_train = mean_absolute_error(flat_true_train, flat_pred_train)
        mae_test  = mean_absolute_error(flat_true_test, flat_pred_test)

        mape_train_val = calculate_mape(flat_true_train, flat_pred_train)
        mape_test_val  = calculate_mape(flat_true_test, flat_pred_test)

        elapsed = time.time() - start_time

        print(f"Results: Train RMSE={rmse_train:.4f}, Test RMSE={rmse_test:.4f}, R2={r2_test:.4f}")

        results.append({
            "Window": w,
            "Forecast_Steps": f,
            "Forecast_Min": forecast_min,
            "Train_RMSE": rmse_train,
            "Test_RMSE": rmse_test,
            "Train_R2": r2_train,
            "Test_R2": r2_test,
            "Train_MAE": mae_train,
            "Test_MAE": mae_test,
            "Train_MAPE": mape_train_val,
            "Test_MAPE": mape_test_val,
            "Time_sec": elapsed
        })

        predictions[(w, f, target)] = {
            "train_true": y_train_true_real.copy(),
            "train_pred": y_train_pred_real.copy(),
            "test_true":  y_test_true_real.copy(),
            "test_pred":  y_test_pred_real.copy(),
            "unit": "m/s"}

        K.clear_session()
        del model, X_train, y_train, X_test, y_test
        gc.collect()

In [ ]:
base_dir = "3-10m"
os.makedirs(base_dir, exist_ok=True)
safe_h = re.sub(r'[^A-Za-z0-9]+', "_", target_h)
for (w, f, target), data in predictions.items():
    w_dir = os.path.join(base_dir, f"w{w}")
    os.makedirs(w_dir, exist_ok=True)

    df_test = pd.DataFrame({
        "True_Test": data["test_true"].flatten(),
        "Pred_Test": data["test_pred"].flatten()
    })

    filename = f"{safe_h}_test_{f*10}min.csv"
    df_test.to_csv(os.path.join(w_dir, filename), index=False)


df_results = pd.DataFrame(results)
for w in df_results["Window"].unique():
    w_dir = os.path.join(base_dir, f"w{w}")
    os.makedirs(w_dir, exist_ok=True)
    df_w = df_results[df_results["Window"] == w]
    df_w.to_csv(os.path.join(w_dir, f"summary_w{w}.csv"), index=False)


colors = ["#006699", "#b30000", "#009933",
          "#ff9900", "#660066", "#666600"]

plt.rcParams["font.size"] = 13

def plot_saved_by_w(predictions_dict, base_dir):

    for idx, ((w, f, target), data) in enumerate(predictions_dict.items()):
        w_dir = os.path.join(base_dir, f"w{w}")
        os.makedirs(w_dir, exist_ok=True)

        true_vals = data["test_true"]
        pred_vals = data["test_pred"]

        rmse = np.sqrt(np.mean((true_vals - pred_vals) ** 2))
        r2 = r2_score(true_vals, pred_vals)

        forecast_min = f * 10

        min_val = min(true_vals.min(), pred_vals.min())
        max_val = max(true_vals.max(), pred_vals.max())

        plt.figure(figsize=(7, 7))
        plt.scatter(
            true_vals,
            pred_vals,
            alpha=0.35,
            color=colors[idx % len(colors)],
            edgecolor="none"
        )

        plt.plot(
            [min_val, max_val],
            [min_val, max_val],
            "r--",
            linewidth=2,
            label="Ideal Fit"
        )

        plt.xlabel("True Wind Speed (m/s)")
        plt.ylabel("Predicted Wind Speed (m/s)")
        plt.title(f"{forecast_min}-Minute Ahead Prediction (w={w})")

        plt.text(
            min_val,
            max_val,
            f"$R^2 = {r2:.4f}$\nRMSE = {rmse:.4f}",
            verticalalignment="top",
            bbox=dict(facecolor="white", alpha=0.85)
        )

        plt.grid(alpha=0.35)
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            os.path.join(w_dir, f"scatter_{forecast_min}min.png"),
            dpi=300
        )
        plt.close()

plot_saved_by_w(predictions, base_dir="3-10m")